# Load library

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as skl
import anndata as ann
import random, os
from scipy.stats import pearsonr as pr
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score as f1
from sklearn.metrics import precision_recall_curve as prc
from sklearn.metrics import silhouette_score as sil
from sklearn.metrics import auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.metrics import silhouette_score
import psutil
import os, sys
import gc
import scipy.sparse as sp
from harmony import harmonize
from tqdm import tqdm
import h5py
from scipy.sparse import csr_matrix
import scipy.io

In [ ]:
sc.set_figure_params(dpi=200)

# General processing functinos

In [ ]:
def whats_memory_eater():
    # Build reverse map of object id -> variable name from globals
    name_map = {id(obj): name for name, obj in globals().items()}

    # Get all tracked objects
    all_objects = gc.get_objects()

    # Safely get size and match variable name
    sizes = []
    for obj in all_objects:
        try:
            size = sys.getsizeof(obj)
            obj_id = id(obj)
            name = name_map.get(obj_id, None)
            sizes.append((size, type(obj), name, repr(obj)[:100]))
        except Exception:
            continue

    # Sort and print top 10
    sizes.sort(reverse=True, key=lambda x: x[0])

    for size, obj_type, name, preview in sizes[:10]:
        print(f"Size: {size / 1024**3} GB | Type: {obj_type} | Name: {name} | Object: {preview}")


In [ ]:
def memory_usgae():
    gc.collect()
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024**3  # in GB

    print(f"Current memory usage: {memory_gb:.2f} GB")

In [ ]:
def reprocess_everything(adata, further_pre = True):
    """
    Reprocess a Scanpy AnnData object using its raw layer (e.g., from a published .h5ad).
    This includes normalization, HVG selection, PCA, neighbors, and UMAP.

    Parameters:
    - adata: AnnData object, must have .raw set

    Returns:
    - Processed AnnData object (modifies in place)
    """

    # Check if raw exists
    if adata.raw is None:
        raise ValueError("AnnData object has no .raw attribute. Cannot proceed with reprocessing.")

    # Extract raw counts
    adata.X = adata.raw.X.copy()
    adata.var = adata.raw.var.copy()
    adata.var_names = adata.raw.var_names.copy()

    # Recalculate mitochondrial content
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print('Standard filtering...')
    # Standard filtering (optional)
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) &
                  (adata.obs['n_genes_by_counts'] <= 5000) &
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    # adata.obs['Project_ID'] = Project_ID
    # adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    if further_pre:
        # Normalize and log transform
        print('Normalizing...')
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # HVG selection
        # sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
        # adata = adata[:, adata.var.highly_variable]
        '''
        sc.pp.highly_variable_genes(
            adata,
            flavor="seurat_v3",  # best for batch-aware HVG selection
            n_top_genes=2000,
            batch_key="Final_sample_id"  # or whatever your batch label column is
        )
        '''
        
        # adata = adata[:, adata.var.highly_variable].copy()

        # Scale
        # print('Scaling...')
        # sc.pp.scale(adata, max_value=10)
        print('Computing PCA...')
        # PCA, neighbors, UMAP
        sc.tl.pca(adata, svd_solver='arpack')
        
        print('Computing neighbors...')
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
        
        print('Computing UMAP...')
        sc.tl.umap(adata)

    print(f"Reprocessed dataset. Final shape: {adata.shape}")
    return adata


# Integrate All data

## Load data

In [ ]:
memory_usgae()

In [ ]:
data_dir = '../../Data/Cancer_cell_data_reprocessed/'
all_h5_files = os.listdir(data_dir)
all_h5_files.sort()

all_h5_files

In [ ]:
cancer_ad_list = []

for h5 in all_h5_files:
    if 'ntegrated' not in h5:
        continue
    if h5.startswith('All') or h5.__contains__('MELA'):
        continue

    print(h5)
    # continue
    # ad = sc.read_h5ad(data_dir + h5)

    cancer_ad_list.append(sc.read_h5ad(data_dir + h5))
    display(cancer_ad_list[-1].to_df())
    print(cancer_ad_list[-1])
    print(cancer_ad_list[-1].raw.shape)

In [ ]:
combined_ad = ann.concat(cancer_ad_list, join="inner", axis=0)
combined_ad

In [ ]:
combined_ad.raw.shape

In [ ]:
memory_usgae()

In [ ]:
del cancer_ad_list
memory_usgae()

In [ ]:
# double check the raw
import scipy
X = combined_ad.raw.X

# Check if sparse
print("Sparse format:", scipy.sparse.issparse(X))

# Summary stats on nonzero values only
nonzero = X.data if scipy.sparse.issparse(X) else X.flatten()

print("Min (nonzero):", nonzero.min())
print("Max (nonzero):", nonzero.max())
print("Mean (nonzero):", nonzero.mean())
print("Dtype:", X.dtype)
print("Fraction of non-integers:", np.mean(nonzero % 1 != 0))
print("Max before normalization:", combined_ad.raw.X.max())


In [ ]:
del X

In [ ]:
combined_ad = reprocess_everything(combined_ad)

In [ ]:
combined_ad

## Unify Annotation

In [ ]:
combined_ad.obs["Final_histological_subtype"].value_counts()

In [ ]:
combined_ad.obs["Final_histological_subtype_backup"] = combined_ad.obs["Final_histological_subtype"]

In [ ]:
def unify_histological_subtype(x):
    val = str(x).lower().strip()

    if "serous" in val and "clear cell" in val:
        return "OVC: Mixed serous + clear cell carcinoma"
    elif "high-grade serous" in val:
        return "OVC: High-grade serous carcinoma"
    elif val.startswith("brca"):
        if "ductal" in val and "lobular" in val:
            return "BRCA: Mixed ductal/lobular carcinoma"
        elif "ductal" in val:
            return "BRCA: Invasive ductal carcinoma"
        elif "lobular" in val:
            return "BRCA: Invasive lobular carcinoma"
        elif "mucinous" in val:
            return "BRCA: Mucinous carcinoma"
        elif "apocrine" in val:
            return "BRCA: Apocrine carcinoma"
        elif "metaplastic" in val:
            return "BRCA: Metaplastic carcinoma"
        elif "metastatic" in val:
            return "BRCA: Metastatic carcinoma"
        elif "unspecified" in val or "malignant neoplasm" in val:
            return "BRCA: Unspecified"
        else:
            return "BRCA: Unspecified"
    elif val.startswith("luca"):
        if "adenocarcinoma" in val:
            return "LUCA: Adenocarcinoma"
        elif "small cell" in val:
            return "LUCA: Small cell carcinoma"
        elif "squamous" in val:
            return "LUCA: Squamous carcinoma"
        elif "pleiomorphic" in val:
            return "LUCA: Pleiomorphic carcinoma"
        elif "large cell" in val:
            return "LUCA: Large cell carcinoma"
        elif "nsclc" in val:
            return "LUCA: Non-small cell lung carcinoma"
        else:
            return "LUCA: Unspecified"
    elif val.startswith("coad"):
        if "mucinous" in val and "neuroendocrine" in val:
            return "COAD: Mucinous neuroendocrine carcinoma"
        elif "mucinous" in val:
            return "COAD: Mucinous adenocarcinoma"
        elif "medullary" in val:
            return "COAD: Medullary carcinoma"
        elif "adenocarcinoma" in val:
            return "COAD: Adenocarcinoma"
        elif "unspecified" in val:
            return "COAD: Unspecified"
        else:
            return "COAD: Unspecified"
    elif val.startswith("mela"):
        return "MELA: Unspecified"
    else:
        # print(x)
        return x

combined_ad.obs["Final_histological_subtype"] = combined_ad.obs["Final_histological_subtype_backup"].apply(unify_histological_subtype)
combined_ad.obs["Final_histological_subtype"].value_counts()

In [ ]:
combined_ad.obs['Final_molecular_subtype'].value_counts()

In [ ]:
combined_ad.obs["Final_tissue"].value_counts()

In [ ]:
combined_ad.obs['Final_tissue_backup'] = combined_ad.obs['Final_tissue']

In [ ]:
def unify_tissue_label(tissue):
    t = str(tissue).lower()
    if t in {"breast", "lung", "liver", "ovary", "brain", "skin", "adrenal"}:
        return t.capitalize()
    elif t in {"colon", "bowel"}:
        return "Colon"
    elif t in {"omentum", "peritoneum", "ascites"}:
        return "Peritoneal"
    elif t in {"pleural effusion"}:
        return "Pleura"
    elif t in {"lymph node", "axilla", "neck"}:
        return "Lymph node"
    elif "soft tissue" in t or "subcutaneous" in t:
        return "Soft tissue"
    elif "bone" in t:
        return "Bone"
    elif "upper quadrant" in t:
        return "Other"
    else:
        return "Other"

combined_ad.obs["Final_tissue"] = combined_ad.obs["Final_tissue_backup"].apply(unify_tissue_label)
combined_ad.obs["Final_tissue"].value_counts()

### Metastasis label (revision3 / BIOINF-2025-2011.R2)

`Final_cancer_type` and `Final_tissue_backup` are both finalized above, which
is everything `label_rules.recompute_metastasis_label()` needs. Assign
`metastasis_label` (No_Mets / Regional_Mets / Distant_Mets) right here rather
than leaving it to whichever downstream script happens to be read next --
previously this column either didn't exist yet at this point in the notebook
or was set by inconsistent per-dataset logic (see
`../Pre-processing/old_label_rule_audit.md`). `label_rules.py` is the single
place that rule lives now; every training/analysis script in this repo calls
the same function on load, so this cell is redundant with them by design --
it just means the *saved atlas* also carries the correct label immediately,
instead of only ever having it patched in downstream.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))  # repo root, so `label_rules` is importable from Pre-processing/
from label_rules import recompute_metastasis_label

recompute_metastasis_label(combined_ad)
combined_ad.obs["metastasis_label"].value_counts()


In [ ]:
combined_ad

In [ ]:
combined_ad.obs.Final_patient_treatment.value_counts()

In [ ]:
combined_ad.obs['Final_patient_treatment_backup'] = combined_ad.obs['Final_patient_treatment']

In [ ]:
treatment_map = {
    'Naïve': 'Naïve',
    'Naive': 'Naïve',
    'Naïve -': 'Naïve',
    'Primary': 'Naïve',
    'NACT/IDS': 'Neoadjuvant chemotherapy',
    'Treated Neoadjuvant AC': 'Neoadjuvant chemotherapy',
    'Treated  Neoadjuvant AC (4x), Paclitaxel (1x)': 'Neoadjuvant chemotherapy',
    'Treated Neoadjuvant AC (4x), Paclitaxel (3x)': 'Neoadjuvant chemotherapy',
    'FOLFOX': 'FOLFOX-based',
    'FOLFOXIRI': 'FOLFOX-based',
    'Endocrine on-treatment': 'Endocrine therapy',
    'Endocrine between treatments': 'Endocrine therapy',
    'Endocrine + Chemotherapy on-treatment': 'Endocrine therapy',
    'Chemotherapy on-treatment': 'Chemotherapy',
    'Chemotherapy between treatments': 'Chemotherapy',
    'Platinum Doublet': 'Platinum-based combination',
    'Platinum Doublet,Immunotherapy': 'Platinum-based combination',
    'Platinum Doublet,PARP inhibitor,TMZ': 'Platinum-based combination',
    'Platinum Doublet,Immunotherapy,TMZ,Other chemotherapy,VEGF inhibitor': 'Platinum-based combination',
    'Platinum Doublet,Immunotherapy,TMZ,Other chemotherapy': 'Platinum-based combination',
    'HER2-targeted between treatments': 'HER2-targeted',
    'Treated AC, Paclitaxel, Herceptin (administered for Dx 3 years prior)': 'HER2-targeted',
    'Immunotherapy between treatments': 'Immunotherapy',
    'distant adjuvant off-treatment': 'Adjuvant off-treatment'
}


In [ ]:
combined_ad.obs['Final_patient_treatment'] = combined_ad.obs['Final_patient_treatment_backup'].map(treatment_map).fillna('Other')
combined_ad.obs.Final_patient_treatment.value_counts()

In [ ]:
combined_ad.obs.Final_patient_stage.value_counts()

In [ ]:
combined_ad.obs['Final_patient_stage_backup'] = combined_ad.obs['Final_patient_stage']

In [ ]:
def unify_stage(stage):
    s = str(stage).strip().upper()

    if s in ["0", "STAGE 0"]:
        return "Stage 0"
    elif any(x in s for x in ["IA", "IB", "I "]):  # include "I " to avoid matching II, III
        return "Stage I"
    elif "IIA" in s or "IIB" in s or s.startswith("II"):
        return "Stage II"
    elif "IIIC" in s or "IIIB" in s or "IIIA" in s or s.startswith("III"):
        return "Stage III"
    elif "IV" in s:
        return "Stage IV"
    elif "UNKNOWN" in s or "NON-CANCER" in s:
        return "Unknown"
    elif "PT" in s:  # heuristic for pathological codes
        if "M1" in s:
            return "Stage IV"
        elif "N2" in s or "N1" in s or "T3" in s or "T4" in s:
            return "Stage III"
        elif "T2" in s:
            return "Stage II"
        elif "T1" in s:
            return "Stage I"
        else:
            return "Unknown"
    else:
        return "Unknown"

# Apply to your AnnData
combined_ad.obs["Final_patient_stage"] = combined_ad.obs["Final_patient_stage_backup"].apply(unify_stage)
combined_ad.obs.Final_patient_stage.value_counts()

In [ ]:
combined_ad.obs['Primary_or_Metastatic'].value_counts()

In [ ]:
combined_ad.obs["Primary_or_Metastatic"] = combined_ad.obs["Primary_or_Metastatic"].replace("Locally advanced", "Primary")


In [ ]:
combined_ad

## Filter low-cell samples

In [ ]:
N = 30  # set your desired minimum cell threshold

# Count cells per sample
sample_counts = combined_ad.obs['Final_sample_id'].value_counts()
print(len(sample_counts), 'total samples')

# Get sample IDs with at least N cells
valid_samples = sample_counts[sample_counts >= N].index
print(len(valid_samples), 'valid samples')


In [ ]:
# Subset AnnData to only keep cells from valid samples
print(combined_ad)
combined_ad = combined_ad[combined_ad.obs['Final_sample_id'].isin(valid_samples)]
print(combined_ad)


In [ ]:
combined_ad.obs['Final_sample_id'].value_counts()

In [ ]:
memory_usgae()

## Plot (Before Integration)

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue']:
    sc.pl.umap(combined_ad, color=obs)

## Harmony integration

In [ ]:
combined_ad

In [ ]:
Z = harmonize(combined_ad.obsm['X_pca'], combined_ad.obs, batch_key = ['Project_ID'])


In [ ]:
combined_ad.obsm['X_pca_harmony_Project_ID'] = Z


In [ ]:
sc.pp.neighbors(combined_ad, use_rep='X_pca_harmony_Project_ID')
sc.tl.umap(combined_ad)

In [ ]:
combined_ad

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 
            'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue',
           'Final_patient_age', 'Final_patient_treatment', 'Final_patient_stage']:
    sc.pl.umap(combined_ad, color=obs)

In [ ]:

combined_ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/All_integrated.harmony.h5ad', compression='gzip')


## Select Hallmark genes

In [ ]:
del combined_ad
memory_usgae()

In [ ]:
ad = sc.read_h5ad('../../Data/Cancer_cell_data_reprocessed/All_integrated.harmony.h5ad')
ad

In [ ]:
def load_hallmark_genes(gmt_file):
    genes = set()
    with open(gmt_file, "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            genes.update(parts[2:])
    return genes

hallmark_genes = load_hallmark_genes("../../h.all.v2024.1.Hs.symbols.gmt")


In [ ]:
# 1. Reset expression to raw counts 
ad.X = ad.raw.X.copy()

print(ad)
# 2. Subset to hallmark genes
ad = ad[:, [gene for gene in ad.var_names if gene in hallmark_genes]].copy()
print(ad)



In [ ]:
import scipy
X = ad.X

# Check if sparse
print("Sparse format:", scipy.sparse.issparse(X))

# Summary stats on nonzero values only
nonzero = X.data if scipy.sparse.issparse(X) else X.flatten()

print("Min (nonzero):", nonzero.min())
print("Max (nonzero):", nonzero.max())
print("Mean (nonzero):", nonzero.mean())
print("Dtype:", X.dtype)
print("Fraction of non-integers:", np.mean(nonzero % 1 != 0))
print("Max before normalization:", ad.X.max())


In [ ]:
memory_usgae()

In [ ]:
del X, nonzero
memory_usgae()

In [ ]:

# 3. Normalize & log1p
sc.pp.normalize_total(ad, target_sum=1e4)
sc.pp.log1p(ad)

# 4. PCA
print("Running PCA...")
sc.tl.pca(ad, svd_solver="arpack", n_comps=40)

# 5. Neighbors (builds connectivities in .obsp)
print("Computing neighbors...")
sc.pp.neighbors(ad, n_neighbors=15, n_pcs=40)

# 6. UMAP
print("Computing UMAP...")
sc.tl.umap(ad)

# Final confirmation
print(f"Final shape: {ad.shape}")
print(f"PCA shape: {ad.obsm['X_pca'].shape}")
print(f"UMAP shape: {ad.obsm['X_umap'].shape}")


In [ ]:
ad.obs["Project_ID"].value_counts()

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 
            'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue',
           'Final_patient_age', 'Final_patient_treatment', 'Final_patient_stage']:
    sc.pl.umap(ad, color=obs)

In [ ]:
from harmony import harmonize
Z = harmonize(ad.obsm['X_pca'], ad.obs, batch_key = ['Project_ID'])
ad.obsm['X_pca_harmony_project_id'] = Z
sc.pp.neighbors(ad, use_rep='X_pca_harmony_project_id')
sc.tl.umap(ad)

In [ ]:
ad.obs["Classifier_label"] = ad.obs["Primary_or_Metastatic"].astype(str) + '_' +ad.obs["Final_tissue"].astype(str)
ad = ad[ad.obs['Classifier_label'] != 'Metastatic_Skin'].copy()

In [ ]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/All_integrated.hallmark.harmony.h5ad', compression='gzip')

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 
            'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue',
           'Final_patient_age', 'Final_patient_treatment', 'Final_patient_stage']:
    sc.pl.umap(ad, color=obs)

In [ ]:
ad = sc.read_h5ad('../../Data/Cancer_cell_data_reprocessed/All_integrated.hallmark.harmony.h5ad')

In [ ]:
ad